In [ ]:
# 04_ramp_shock_baseline.ipynb -- LightGBM baseline for the 1-4-slot-lead-time
# ramp-shock target (same feature pipeline as 03_violation_baseline.ipynb)
# !pip install lightgbm -q

import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.metrics import (average_precision_score, f1_score, precision_score,
                              recall_score, precision_recall_curve)

import features as f

TARGET = "ramp_lead"

scada = pd.read_csv("data/study2_scada.csv", parse_dates=["date"])
scada = scada.sort_values(["date", "time"]).reset_index(drop=True)
feat_df = f.build_feature_table(scada)

df = feat_df.dropna(subset=[TARGET]).copy()

# --- Time-aware split (same as 03_violation_baseline.ipynb) ---
train = df[(df["date"] >= "2024-11-04") & (df["date"] <= "2025-06-30")]
val = df[(df["date"] >= "2025-07-01") & (df["date"] <= "2025-12-31")]
test = df[df["date"] >= "2026-01-01"]

print("train:", train.shape, "val:", val.shape, "test:", test.shape)
print("event rate train/val/test:", train[TARGET].mean(), val[TARGET].mean(), test[TARGET].mean())

X_train, y_train = train[f.FEATURE_COLS], train[TARGET]
X_val, y_val = val[f.FEATURE_COLS], val[TARGET]
X_test, y_test = test[f.FEATURE_COLS], test[TARGET]

# NOTE (2026-07-11): NOT using scale_pos_weight -- see features.py's
# scale_pos_weight() docstring and 03_violation_baseline.ipynb for the full story.
model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbosity=-1)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="average_precision",
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
)
print(f"\nbest_iteration_: {model.best_iteration_}")

proba_val = model.predict_proba(X_val)[:, 1]
proba_test = model.predict_proba(X_test)[:, 1]
pr_auc = average_precision_score(y_test, proba_test)
base_rate = y_test.mean()
print(f"PR-AUC: {pr_auc:.4f}  (random baseline = base rate = {base_rate:.4f})")

preds_05 = (proba_test >= 0.5).astype(int)
print(f"F1 @ 0.5 threshold: {f1_score(y_test, preds_05):.4f}")

# --- Threshold selection done on VAL, then frozen and applied to TEST (fixed
# 2026-07-14, same methodology fix as 03_violation_baseline.ipynb: previously the
# "best-F1" threshold was picked by scanning TEST's own PR curve for its best point --
# reporting the best-in-hindsight operating point rather than what a threshold fixed in
# advance would actually achieve on unseen data. PR-AUC and F1@0.5 are unaffected.) ---
precision_val, recall_val, thresh_val = precision_recall_curve(y_val, proba_val)
f1s_val = 2 * precision_val * recall_val / (precision_val + recall_val + 1e-12)
best_idx_val = np.nanargmax(f1s_val[:-1])
best_thresh = thresh_val[best_idx_val]

preds_best = (proba_test >= best_thresh).astype(int)
test_precision = precision_score(y_test, preds_best, zero_division=0)
test_recall = recall_score(y_test, preds_best)
test_f1 = f1_score(y_test, preds_best)
print(f"Best-F1 threshold (selected on VAL): {best_thresh:.4f} -- applied to TEST: "
      f"F1={test_f1:.4f} (precision={test_precision:.4f}, recall={test_recall:.4f})")

idx95_val = np.where(precision_val[:-1] >= 0.95)[0]
if len(idx95_val):
    thresh_95 = thresh_val[idx95_val[-1]]  # laxest VAL threshold still hitting >=95% precision on VAL
    preds_95 = (proba_test >= thresh_95).astype(int)
    print(f"Threshold for >=95% precision (selected on VAL): {thresh_95:.4f} -- applied to TEST: "
          f"precision={precision_score(y_test, preds_95, zero_division=0):.4f}, "
          f"recall={recall_score(y_test, preds_95):.4f}")
else:
    print("No VAL threshold reaches >=95% precision")

# --- PR curve on TEST (the curve itself doesn't select a threshold, so no leakage
# concern here -- only marking a point on it does) ---
precision, recall, thresh = precision_recall_curve(y_test, proba_test)

plt.figure(figsize=(6, 6))
plt.plot(recall, precision, color="#2a78d6", linewidth=2)
plt.axhline(base_rate, color="#898781", linewidth=1, linestyle="--",
            label=f"random baseline ({base_rate:.3f})")
plt.scatter([test_recall], [test_precision], color="#0b0b0b", zorder=5,
            label="best-F1 point (threshold from VAL)", s=40)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-recall curve -- ramp-shock lead-time classifier")
plt.legend()
plt.tight_layout()
plt.show()

tp = int(((preds_best == 1) & (y_test == 1)).sum())
fp = int(((preds_best == 1) & (y_test == 0)).sum())
fn = int(((preds_best == 0) & (y_test == 1)).sum())
tn = int(((preds_best == 0) & (y_test == 0)).sum())
cm = np.array([[tn, fp], [fn, tp]])

fig, ax = plt.subplots(figsize=(6, 4.5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], ["pred no-event", "pred event"])
ax.set_yticks([0, 1], ["actual no-event", "actual event"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_title(f"Confusion matrix @ best-F1 threshold ({best_thresh:.3f}, from VAL)", fontsize=9)
plt.tight_layout()
plt.show()

importance = pd.Series(model.feature_importances_, index=f.FEATURE_COLS).sort_values(ascending=False)
print("\nTop 15 features:\n", importance.head(15))

plt.figure(figsize=(8, 6))
top_imp = importance.head(15)
norm = mpl.colors.Normalize(vmin=top_imp.min(), vmax=top_imp.max())
colors = mpl.colormaps["cool"](norm(top_imp.values))
plt.barh(top_imp.index[::-1], top_imp.values[::-1], color=colors[::-1])
plt.xlabel("Split count")
plt.title("Feature importance -- ramp-shock lead-time classifier")
plt.tight_layout()
plt.show()

# --- Results (re-verified 2026-07-14, FIFTH pass -- methodology fix only, see the note
#     above the threshold-selection block: best-F1 threshold and the >=95%-precision
#     threshold are now chosen on VAL and frozen before being applied to TEST, instead
#     of being picked in hindsight on TEST's own curve.) ---
# PR-AUC: unchanged at 0.7486 (no feature/model change -- PR-AUC never involved
# threshold selection, so this number was never affected by the bug).
# Best-F1 threshold from VAL, applied to TEST: see the printed output for this run's
# exact precision/recall -- expect these to be close to, but not identical to, the old
# in-hindsight numbers (F1=0.6803, precision=63.3%, recall=73.5%), since VAL and TEST
# cover different months and the val-selected threshold is very unlikely to be exactly
# TEST's own optimum.
#
# Full history of this notebook's numbers: (1) PR-AUC 0.7140 with scale_pos_weight;
# (2) 0.7248 after removing it + adding solar_delta_mw/solar_roll8_std; (3) 0.7446 after
# removing share_res_pct + 11 corridor/cross-border columns (whole-day aggregates
# broadcast to every slot -- tested as a leakage concern, found to be pure noise
# instead); (4) 0.7486, after adding freq_hz_delta and wind_delta_mw (the two
# next-highest correlations with violation_lead from the diagnostic scan that
# originally found solar_delta_mw). Smaller gain here than for violation_lead (see
# 03_violation_baseline.ipynb, +32% PR-AUC there) -- wind_delta_mw contributes real,
# nonzero importance (rank 12) but freq_hz_delta barely registers for this target,
# unlike for violation. Makes sense: this target is about DEMAND swings, and freq_hz's
# own volatility is a more direct signal for frequency-adjacent problems (violation)
# than for demand-driven ones (ramp-shock).
#  (5) THIS version -- PR-AUC unchanged (0.7486); operating-point (best-F1, >=95%-
# precision) numbers are now honestly out-of-sample instead of hindsight-optimal.
#
# Top features: hour, demand_delta_mw, month, demand_met_mw_lag3, solar_delta_mw,
# solar_roll8_std -- solar and demand-trajectory features dominate, consistent with
# 01_eda.ipynb's sunrise/sunset clustering finding. Corridor columns no longer appear at
# all (removed from FEATURE_COLS) -- Era 2's daily-resolution corridor-flow finding
# remains a separate, valid, unaffected result, it just isn't what drives this live
# classifier.
#
# This target remains markedly easier to predict with lead time than frequency
# violation (PR-AUC 0.1567 even after the same rounds of fixes) -- plausibly because a
# ramp-shock is a direct, mechanical property of the demand/generation trajectory
# itself, while a frequency violation is a downstream consequence that depends on how
# well AGC/reserves absorb a given ramp, adding a layer of noise the raw features here
# don't fully capture.
